<a href="https://colab.research.google.com/github/deepakri201/SR_for_NLST_Sybil/blob/main/explore_metadata/NLST_Sybil_explore_metadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explore metadata

In this notebook, we explore the metadata from the DICOM Structured Reports joined with clinical metadata from the NLST collection in the IDC tables.

Since we can't handle when there are more than 1 bounding box on a single slice - for joining with IDC metadata, we remove these cases from Part A. For Part B we include everything.

1. Download metadata from SR table from github
2. Join with NLST clincal metadata
  - lung lobe location
  - IDC-03 codes
  - lung staging classification
3. Add the OHIF urls

Deepa Krishnaswamy

Brigham and Women's Hospital

September 2025

# Parameterization

In [ ]:
#@title Enter your Project ID here
# initialize this variable with your Google Cloud Project ID!
project_name = "idc-external-018" #@param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = project_name

!gcloud config set project $project_name

from google.colab import auth
auth.authenticate_user()

Updated property [core/project].


# Environment setup

In [ ]:
!pip install idc-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 106.8 MB/s eta 0:00:00
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.3.2
    Uninstalling duckdb-1.3.2:
      Successfully uninstalled duckdb-1.3.2


In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

import json
from pathlib import Path

In [ ]:
from google.cloud import bigquery
from google.cloud import storage

In [ ]:
import datetime

In [ ]:
from idc_index import IDCClient

idc_client = IDCClient.client()

In [ ]:
# Get the bbox_measurements csv file from github
# !wget https://github.com/deepakri201/SR_for_NLST_Sybil/releases/download/v1.0.1/bbox_measurements.csv


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


# A. Join SR bounding boxes with clinical metadata from IDC - only keeping where the bounding boxes overlap with the clincal metadata

## df_sr - get the metadata from the SRs

In [ ]:
# df_sr = pd.read_csv("/content/bbox_measurements.csv") # later, query instead.

# Query from BQ instead
client_bq = bigquery.Client(project=project_name)
query = f"""
    SELECT
      *
    FROM
      `idc-external-018.sr_nlst_sybil.bbox_measurements`
      """
df_sr = client_bq.query(query).to_dataframe()

In [ ]:
# First add columns for the width, height, center_x, and center_y

width_list = []
height_list = []
center_x_list = []
center_y_list = []

for index, row in df_sr.iterrows():
  # Get values
  x0 = row['x0']; y0 = row['y0']
  x1 = row['x1']; y1 = row['y1']
  x2 = row['x2']; y2 = row['y2']
  x3 = row['x3']; y3 = row['y3']
  # calculate the width, height and center, as these are needed for display
  min_x = np.min([x0, x1, x2, x3]) # using roi.GraphicData: min_x = np.min([bbox[0], bbox[2], bbox[4], bbox[6]])
  max_x = np.max([x0, x2, x2, x3]) # using roi.GraphicData: max_x = np.max([bbox[0], bbox[2], bbox[4], bbox[6]])
  min_y = np.min([y0, y1, y2, y3]) # using roi.GraphicData: min_y = np.min([bbox[1], bbox[3], bbox[5], bbox[7]])
  max_y = np.max([y0, y1, y2, y3]) # using roi.GraphicData: max_y = np.max([bbox[1], bbox[3], bbox[5], bbox[7]])
  width = max_x - min_x
  height = max_y - min_y
  center_x = min_x + width/2
  center_y = min_y + height/2
  # append
  width_list.append(width)
  height_list.append(height)
  center_x_list.append(center_x)
  center_y_list.append(center_y)

# Add columns
df_sr['width'] = width_list
df_sr['height'] = height_list
df_sr['center_x'] = center_x_list
df_sr['center_y'] = center_y_list

df_sr.head()

,PatientID,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,ReferencedSeriesInstanceUID,trackingIdentifier,trackingUniqueIdentifier,finding,findingSite,ReferencedSOPInstanceUID,...,x1,y1,x2,y2,x3,y3,width,height,center_x,center_y
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,1,1.2.826.0.1.3680043.8.498.89537880202650470788...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.15782756894992033517041215...,...,221.213928,285.038818,221.213928,324.776123,176.891541,324.776123,44.322388,39.737305,199.052734,304.907471
1,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,2,1.2.826.0.1.3680043.8.498.38187982882062875892...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.20401816777450596174524730...,...,220.000000,287.000000,220.000000,326.000000,177.000000,326.000000,43.000000,39.000000,198.500000,306.500000
2,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,3,1.2.826.0.1.3680043.8.498.79754527055430472666...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.32049018332506010968782808...,...,220.000000,289.000000,220.000000,327.000000,178.000000,327.000000,42.000000,38.000000,199.000000,308.000000
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,4,1.2.826.0.1.3680043.8.498.60746621313354591972...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.23447016361362557671014824...,...,220.000000,292.000000,220.000000,329.000000,179.000000,329.000000,41.000000,37.000000,199.500000,310.500000
4,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,5,1.2.826.0.1.3680043.8.498.58936828276024984484...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.29202164341698057454176258...,...,219.000000,294.000000,219.000000,329.000000,179.000000,329.000000,40.000000,35.000000,199.000000,311.500000


In [ ]:
# We need the following fields in order to convert from pixel coordintes to mm for the bounding box
# And we need the Dimensions, Pixel spacing IPP, especially IPP[2] for the z value

referenced_sop_instance_uid_list = list(df_sr['ReferencedSOPInstanceUID'].values)

client_bq = bigquery.Client(project=project_name)

query = f"""
    SELECT
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      SOPInstanceUID,
      `Rows` as num_rows,
      `Columns` as num_columns,
      PixelSpacing,
      ImagePositionPatient
    FROM
      `bigquery-public-data.idc_current.dicom_all`
    WHERE
      SOPInstanceUID IN UNNEST(@referenced_sop_instance_uid_list)
    ORDER BY
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      ImagePositionPatient[SAFE_OFFSET(2)]
      """

job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("referenced_sop_instance_uid_list", "STRING", referenced_sop_instance_uid_list)])
df_idc = client_bq.query(query, job_config=job_config).to_dataframe()

In [ ]:
# Reformat the PixelSpacing and the ImagePositionPatient columns

df_idc['pixel_spacing_x'] = [np.float32(f[0]) for f in df_idc['PixelSpacing'].values]
df_idc['pixel_spacing_y'] = [np.float32(f[1]) for f in df_idc['PixelSpacing'].values]
df_idc['ipp0'] = [np.float32(f[0]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp1'] = [np.float32(f[1]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp2'] = [np.float32(f[2]) for f in df_idc['ImagePositionPatient'].values]

df_idc = df_idc[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID',
                 'num_rows', 'num_columns',
                 'pixel_spacing_x', 'pixel_spacing_y',
                 'ipp0', 'ipp1', 'ipp2']]

In [ ]:
df_idc.head()

,PatientID,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,num_rows,num_columns,pixel_spacing_x,pixel_spacing_y,ipp0,ipp1,ipp2
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29991037322734048580038819...,512,512,0.585938,0.585938,-149.707031,-319.707031,-76.400002
1,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.27443508115501826384206327...,512,512,0.585938,0.585938,-149.707031,-319.707031,-78.400002
2,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.16899951679153198601142574...,512,512,0.585938,0.585938,-149.707031,-319.707031,-80.400002
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.17481124987277919843449170...,512,512,0.585938,0.585938,-149.707031,-319.707031,-82.400002
4,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29574962348509387538142601...,512,512,0.585938,0.585938,-149.707031,-319.707031,-84.400002


## df_nlst_metadata - get the associated clinical metadata - for classification

In [ ]:
# # Here we save an intermediate table with the PatientID, SeriesInstanceUID and SOPInstanceUID
# # This will help us to get the associated NLST clinical metadata

# client = bigquery.Client(project=project_name, location='US') # since below can't mix US and us-central1
# df_needed = df_idc[['PatientID', 'SeriesInstanceUID', 'SOPInstanceUID']].drop_duplicates()
# table_id = ".".join([project_name,
#                      dataset_name,
#                      "needed_uids"])
# job_config = bigquery.LoadJobConfig(
#         write_disposition=bigquery.job.WriteDisposition.WRITE_TRUNCATE
#     )
# client.load_table_from_dataframe(df_needed, table_id, job_config=job_config).result()

In [ ]:
PatientID_list = sorted(list(set(df_idc['PatientID'].values)))
PatientID_list = [str(f) for f in PatientID_list]
print(len(PatientID_list))

SeriesInstanceUID_list = list(set(df_idc['SeriesInstanceUID'].values))
print(len(SeriesInstanceUID_list))

SOPInstanceUID_list = list(set(df_idc['SOPInstanceUID'].values))
print(len(SOPInstanceUID_list))

581
970
9222


In [ ]:
# Here we get the staging data

query = f"""
WITH dicom_mapped AS (
  SELECT
    PatientID,
    StudyInstanceUID,
    StudyDate,
    SeriesInstanceUID,
    SOPInstanceUID,
    InstanceNumber,
    `Rows`,
    `Columns`,
    CASE StudyDate
      WHEN '1999-01-02' THEN 0
      WHEN '2000-01-02' THEN 1
      WHEN '2001-01-02' THEN 2
      ELSE 3
    END AS StudyDate_mapped,
    COUNT(*) OVER (PARTITION BY SeriesInstanceUID) AS sop_count_per_series
  FROM `bigquery-public-data.idc_current.dicom_all`
  WHERE SeriesInstanceUID IN UNNEST(@SeriesInstanceUID_list)
)

SELECT
  dicom_mapped.PatientID AS PatientID,
  dicom_mapped.StudyInstanceUID,
  dicom_mapped.StudyDate,
  dicom_mapped.SeriesInstanceUID,
  dicom_mapped.SOPInstanceUID,
  ctab.sct_slice_num,
  ctab.sct_epi_loc,
  ctab.sct_margins,
  ctab.sct_pre_att,
  ctab.study_yr,
  dicom_mapped.Rows,
  dicom_mapped.Columns,
  dicom_mapped.sop_count_per_series,
  prsn.de_stag,
  CAST(prsn.de_type AS STRING) as de_type
FROM
  `bigquery-public-data.idc_current_clinical.nlst_ctab` AS ctab
JOIN
  `bigquery-public-data.idc_current_clinical.nlst_prsn` AS prsn
  ON prsn.dicom_patient_id = ctab.dicom_patient_id
JOIN
  dicom_mapped
ON dicom_mapped.InstanceNumber = ctab.sct_slice_num
   AND ctab.study_yr = dicom_mapped.StudyDate_mapped
   AND dicom_mapped.PatientID = ctab.dicom_patient_id

"""
job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("SOPInstanceUID_list", "STRING", SOPInstanceUID_list),
                                                       bigquery.ArrayQueryParameter("SeriesInstanceUID_list", "STRING", SeriesInstanceUID_list)])
df_nlst_metadata = client_bq.query(query, job_config=job_config).to_dataframe()

In [ ]:
df_nlst_metadata.head()

,PatientID,StudyInstanceUID,StudyDate,SeriesInstanceUID,SOPInstanceUID,sct_slice_num,sct_epi_loc,sct_margins,sct_pre_att,study_yr,Rows,Columns,sop_count_per_series,de_stag,de_type
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.23447016361362557671014824...,38,1,2,1,0,512,512,162,110,8140
1,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1.2.840.113654.2.55.50761756412482430061802871...,1.2.840.113654.2.55.23050894131215950278743827...,39,1,1,2,1,512,512,157,110,8140
2,100147,1.2.840.113654.2.55.13303292650860633016545772...,1999-01-02,1.2.840.113654.2.55.24785488463405747713776937...,1.2.840.113654.2.55.17981552983033538571417359...,88,1,3,2,0,512,512,110,110,8140
3,100147,1.2.840.113654.2.55.31958452963320032523273261...,2000-01-02,1.2.840.113654.2.55.15708941008648745210499888...,1.2.840.113654.2.55.19851369674160913596156868...,92,1,3,2,1,512,512,116,110,8140
4,100158,1.2.840.113654.2.55.81185422866512279860334872...,2001-01-02,1.2.840.113654.2.55.31060976780967844152296392...,1.2.840.113654.2.55.18324898484332559595555356...,57,1,2,1,2,512,512,146,110,8140


In [ ]:
len(df_nlst_metadata)

1405

In [ ]:
df_nlst_metadata.to_csv("/content/nlst_metadata.csv")

## Join the tables to hold the SR info and clinical metadata info

In [ ]:
# We have to remove any ReferencedSOPInstanceUIDs in df_sr that have more than 1 bounding box on the slice
# Since we can't join the metadata for these - could be two on the same slice and in the same lobe

print(len(list(set(df_sr['ReferencedSOPInstanceUID'].values)))) # 9222
print(len(df_sr)) # 9280

df_sr = df_sr.drop_duplicates(subset=['ReferencedSOPInstanceUID'], keep=False)
print(len(list(set(df_sr['ReferencedSOPInstanceUID'].values))))
print(len(df_sr))

9222
9280
9166
9166


In [ ]:
# Then join with df_idc
df_sr_join = df_sr.merge(df_idc,
                         left_on=['ReferencedSOPInstanceUID'],
                         right_on=['SOPInstanceUID'],
                         suffixes=('','_right'))
# Drop the duplicate column from the right dataframe
df_sr_join = df_sr_join.drop(columns=['PatientID_right','SOPInstanceUID_right'])

# Then join with the df_nlst_metadata
df_sr_and_nlst = df_sr_join.merge(df_nlst_metadata,
                                  left_on=['ReferencedSOPInstanceUID'],
                                  right_on=['SOPInstanceUID'],
                                  suffixes=('','_right'))
df_sr_and_nlst = df_sr_and_nlst.drop(columns=['PatientID_right','StudyInstanceUID_right','SeriesInstanceUID_right','num_rows', 'num_columns'])
# Rename columns
# df_sr_and_nlst = df_sr_and_nlst.rename({'trackingIdentifier': 'TrackingIdentifier',
#                                         'trackingUniqueIdentifier':'TrackingUID',
#                                         'finding':'FindingType',
#                                         'findingSite': 'FindingSite',
#                                         'ReferencedSOPInstanceUID':'SOPInstanceUID'}, axis=1)
df_sr_and_nlst = df_sr_and_nlst.rename({'trackingIdentifier': 'TrackingIdentifier',
                                        'trackingUniqueIdentifier':'TrackingUID',
                                        'finding':'FindingType',
                                        'findingSite': 'FindingSite'}, axis=1)
# Reorder the columns
df_sr_and_nlst = df_sr_and_nlst[['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr', 'SeriesInstanceUID', 'ReferencedSeriesInstanceUID', 'sop_count_per_series',
                                 'TrackingIdentifier', 'TrackingUID', 'ReferencedSOPInstanceUID',
                                 'FindingType', 'FindingSite',
                                 'pixel_spacing_x', 'pixel_spacing_y',
                                 'width', 'height', 'center_x', 'center_y', 'ipp0', 'ipp1', 'ipp2',
                                 'sct_slice_num', 'sct_epi_loc', 'sct_margins', 'sct_pre_att', 'de_stag', 'de_type']]
# Order the values in the columns
df_sr_and_nlst = df_sr_and_nlst.sort_values(by=['PatientID', 'study_yr', 'StudyInstanceUID', 'ReferencedSeriesInstanceUID', 'TrackingIdentifier'])
df_sr_and_nlst.head()



,PatientID,StudyInstanceUID,StudyDate,study_yr,SeriesInstanceUID,ReferencedSeriesInstanceUID,sop_count_per_series,TrackingIdentifier,TrackingUID,ReferencedSOPInstanceUID,...,center_y,ipp0,ipp1,ipp2,sct_slice_num,sct_epi_loc,sct_margins,sct_pre_att,de_stag,de_type
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,4,1.2.826.0.1.3680043.8.498.60746621313354591972...,1.2.840.113654.2.55.23447016361362557671014824...,...,310.5,-149.707031,-319.707031,-92.400002,38,1,2,1,110,8140
1,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1,1.2.826.0.1.3680043.8.498.83730668932055449360...,1.2.840.113654.2.55.50761756412482430061802871...,157,6,1.2.826.0.1.3680043.8.498.45402182495842328878...,1.2.840.113654.2.55.23050894131215950278743827...,...,328.5,-133.726562,-309.726562,1220.699951,39,1,1,2,110,8140
3,100147,1.2.840.113654.2.55.13303292650860633016545772...,1999-01-02,0,1.2.826.0.1.3680043.8.498.50951503649235744962...,1.2.840.113654.2.55.24785488463405747713776937...,110,3,1.2.826.0.1.3680043.8.498.36065263653997571352...,1.2.840.113654.2.55.17981552983033538571417359...,...,313.5,-177.300003,-169.000000,-51.189999,88,1,3,2,110,8140
2,100147,1.2.840.113654.2.55.31958452963320032523273261...,2000-01-02,1,1.2.826.0.1.3680043.8.498.18532751859109067988...,1.2.840.113654.2.55.15708941008648745210499888...,116,5,1.2.826.0.1.3680043.8.498.12309181670332046684...,1.2.840.113654.2.55.19851369674160913596156868...,...,332.5,-165.000000,-188.899994,-49.275002,92,1,3,2,110,8140
4,100158,1.2.840.113654.2.55.81185422866512279860334872...,2001-01-02,2,1.2.826.0.1.3680043.8.498.17370077098876814230...,1.2.840.113654.2.55.31060976780967844152296392...,146,4,1.2.826.0.1.3680043.8.498.78001958736076182639...,1.2.840.113654.2.55.18324898484332559595555356...,...,272.0,-179.100006,-175.000000,-149.440002,57,1,2,1,110,8140


In [ ]:
len(df_sr_and_nlst)

782

In [ ]:
df_sr_and_nlst.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr',
       'SeriesInstanceUID', 'ReferencedSeriesInstanceUID',
       'sop_count_per_series', 'TrackingIdentifier', 'TrackingUID',
       'ReferencedSOPInstanceUID', 'FindingType', 'FindingSite',
       'pixel_spacing_x', 'pixel_spacing_y', 'width', 'height', 'center_x',
       'center_y', 'ipp0', 'ipp1', 'ipp2', 'sct_slice_num', 'sct_epi_loc',
       'sct_margins', 'sct_pre_att', 'de_stag', 'de_type'],
      dtype='object')

## Calculate extra features

In [ ]:
segments_code_mapping_df = pd.read_csv(
    "https://raw.githubusercontent.com/deepakri201/SEG_and_SR_for_NLSTSeg/main/NLSTSeg_icd03_codes.csv"
)

In [ ]:
segments_code_mapping_df = segments_code_mapping_df[['SegmentedPropertyTypeCodeSequence.CodeValue','SegmentedPropertyTypeCodeSequence.CodeMeaning']]
segments_code_mapping_df = segments_code_mapping_df.rename(columns={'SegmentedPropertyTypeCodeSequence.CodeValue': 'ICD03',
                                                                    'SegmentedPropertyTypeCodeSequence.CodeMeaning': 'Description'})
segments_code_mapping_df['ICD03'] = [str(f) for f in segments_code_mapping_df['ICD03'].values]

In [ ]:
segments_code_mapping_df

,ICD03,Description
0,8481,Mucin-producing adenocarcinoma
1,8490,Signet ring cell carcinoma
2,8071,"Squamous cell carcinoma, keratinizing"
3,8252,"Bronchiolo-alveolar carcinoma, non-mucinous"
4,8570,Adenocarcinoma with squamous metaplasia
5,8323,Mixed cell adenocarcinoma
6,8010,Carcinoma in situ
7,8050,Papillary adenocarcinoma
8,8042,Oat cell carcinoma
9,8254,"Bronchiolo-alveolar carcinoma, mixed mucinous ..."


In [ ]:
de_stag_mapping = {'110': "Stage IA",
                   '120': "Stage IB",
                   '210': "Stage IIA",
                   '220': "Stage IIB",
                   '310': "Stage IIIA",
                   '320': "Stage IIIB",
                   '400': "Stage IV",
                   '888': "TNM not available",
                   '900': "Occult Carcinoma",
                   '994': "Carcinoid, cannot be assessed",
                   '999': "Unknown, cannot be assessed"}
# de_type_mapping = segments_code_mapping_df.to_dict(orient='dict')
de_type_mapping = dict(zip(segments_code_mapping_df.ICD03,segments_code_mapping_df.Description))
sct_epi_loc_mapping = {'1': "Right Upper Lobe" ,
                       '2': "Right Middle Lobe",
                       '3': "Right Lower Lobe",
                       '4': "Left Upper Lobe",
                       '5': "Lingula",
                       '6': "Left Lower Lobe",
                       '8': "Other (Specify in comments)"}
sct_margins_mapping = {'1': "Spiculated (Stellate)",
                       '2': "Smooth",
                       '3': "Poorly defined",
                       '9': "Unable to determine"}
sct_pre_att_mapping = {'.M': "Missing",
                       'N': "Not applicable (sct_ab_desc is not 51)",
                       '1': "Soft Tissue",
                       '2': "Ground glass",
                       '3': "Mixed",
                       '4': "Fluid/water",
                       '6': "Fat",
                       '7': "Other",
                       '9': "Unable to determine"}

In [ ]:
df_sr_and_nlst['de_stag_mapping'] = df_sr_and_nlst['de_stag'].map(de_stag_mapping)
df_sr_and_nlst['de_type_mapping'] = df_sr_and_nlst['de_type'].map(de_type_mapping)
df_sr_and_nlst['sct_epi_loc_mapping'] = df_sr_and_nlst['sct_epi_loc'].map(sct_epi_loc_mapping)
df_sr_and_nlst['sct_margins_mapping'] = df_sr_and_nlst['sct_margins'].map(sct_margins_mapping)
df_sr_and_nlst['sct_pre_att_mapping'] = df_sr_and_nlst['sct_pre_att'].map(sct_pre_att_mapping)


In [ ]:
df_sr_and_nlst.head()

,PatientID,StudyInstanceUID,StudyDate,study_yr,SeriesInstanceUID,ReferencedSeriesInstanceUID,sop_count_per_series,TrackingIdentifier,TrackingUID,ReferencedSOPInstanceUID,...,sct_epi_loc,sct_margins,sct_pre_att,de_stag,de_type,de_stag_mapping,de_type_mapping,sct_epi_loc_mapping,sct_margins_mapping,sct_pre_att_mapping
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,4,1.2.826.0.1.3680043.8.498.60746621313354591972...,1.2.840.113654.2.55.23447016361362557671014824...,...,1,2,1,110,8140,Stage IA,Adenocarcinoma,Right Upper Lobe,Smooth,Soft Tissue
1,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1,1.2.826.0.1.3680043.8.498.83730668932055449360...,1.2.840.113654.2.55.50761756412482430061802871...,157,6,1.2.826.0.1.3680043.8.498.45402182495842328878...,1.2.840.113654.2.55.23050894131215950278743827...,...,1,1,2,110,8140,Stage IA,Adenocarcinoma,Right Upper Lobe,Spiculated (Stellate),Ground glass
3,100147,1.2.840.113654.2.55.13303292650860633016545772...,1999-01-02,0,1.2.826.0.1.3680043.8.498.50951503649235744962...,1.2.840.113654.2.55.24785488463405747713776937...,110,3,1.2.826.0.1.3680043.8.498.36065263653997571352...,1.2.840.113654.2.55.17981552983033538571417359...,...,1,3,2,110,8140,Stage IA,Adenocarcinoma,Right Upper Lobe,Poorly defined,Ground glass
2,100147,1.2.840.113654.2.55.31958452963320032523273261...,2000-01-02,1,1.2.826.0.1.3680043.8.498.18532751859109067988...,1.2.840.113654.2.55.15708941008648745210499888...,116,5,1.2.826.0.1.3680043.8.498.12309181670332046684...,1.2.840.113654.2.55.19851369674160913596156868...,...,1,3,2,110,8140,Stage IA,Adenocarcinoma,Right Upper Lobe,Poorly defined,Ground glass
4,100158,1.2.840.113654.2.55.81185422866512279860334872...,2001-01-02,2,1.2.826.0.1.3680043.8.498.17370077098876814230...,1.2.840.113654.2.55.31060976780967844152296392...,146,4,1.2.826.0.1.3680043.8.498.78001958736076182639...,1.2.840.113654.2.55.18324898484332559595555356...,...,1,2,1,110,8140,Stage IA,Adenocarcinoma,Right Upper Lobe,Smooth,Soft Tissue


In [ ]:
df_sr_and_nlst.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr',
       'SeriesInstanceUID', 'ReferencedSeriesInstanceUID',
       'sop_count_per_series', 'TrackingIdentifier', 'TrackingUID',
       'ReferencedSOPInstanceUID', 'FindingType', 'FindingSite',
       'pixel_spacing_x', 'pixel_spacing_y', 'width', 'height', 'center_x',
       'center_y', 'ipp0', 'ipp1', 'ipp2', 'sct_slice_num', 'sct_epi_loc',
       'sct_margins', 'sct_pre_att', 'de_stag', 'de_type', 'de_stag_mapping',
       'de_type_mapping', 'sct_epi_loc_mapping', 'sct_margins_mapping',
       'sct_pre_att_mapping'],
      dtype='object')

In [ ]:
# Need to split the Finding and FindingSite into columns for the CodeValue, CodingSchemeDesignator, and CodeMeaning, in order to save as csv and be able to load as a BQ table.
# df_expanded = pd.json_normalize(df_sr_and_nlst["FindingType"].apply(json.loads))
# df_sr_and_nlst = df_sr_and_nlst.join(df_expanded)

# df_expanded = pd.json_normalize(df_sr_and_nlst["FindingSite"].apply(json.loads))
# df_sr_and_nlst = df_sr_and_nlst.join(df_expanded)

df_expanded = pd.json_normalize(df_sr_and_nlst["FindingType"])
df_sr_and_nlst = df_sr_and_nlst.join(df_expanded, lsuffix="FindingType")
df_sr_and_nlst = df_sr_and_nlst.rename(columns = {'CodeValue': 'FindingType_CodeValue',
                                                  'CodingSchemeDesignator': 'FindingType_CodingSchemeDesignator',
                                                  'CodeMeaning': 'FindingType_CodeMeaning'})

df_expanded = pd.json_normalize(df_sr_and_nlst["FindingSite"])
df_sr_and_nlst = df_sr_and_nlst.join(df_expanded, lsuffix="FindingSite")
df_sr_and_nlst = df_sr_and_nlst.rename(columns = {'CodeValue': 'FindingSite_CodeValue',
                                                  'CodingSchemeDesignator': 'FindingSite_CodingSchemeDesignator',
                                                  'CodeMeaning': 'FindingSite_CodeMeaning'})


In [ ]:
df_sr_and_nlst.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr',
       'SeriesInstanceUID', 'ReferencedSeriesInstanceUID',
       'sop_count_per_series', 'TrackingIdentifier', 'TrackingUID',
       'ReferencedSOPInstanceUID', 'FindingType', 'FindingSite',
       'pixel_spacing_x', 'pixel_spacing_y', 'width', 'height', 'center_x',
       'center_y', 'ipp0', 'ipp1', 'ipp2', 'sct_slice_num', 'sct_epi_loc',
       'sct_margins', 'sct_pre_att', 'de_stag', 'de_type', 'de_stag_mapping',
       'de_type_mapping', 'sct_epi_loc_mapping', 'sct_margins_mapping',
       'sct_pre_att_mapping', 'FindingType_CodeValue',
       'FindingType_CodingSchemeDesignator', 'FindingType_CodeMeaning',
       'FindingSite_CodeValue', 'FindingSite_CodingSchemeDesignator',
       'FindingSite_CodeMeaning'],
      dtype='object')

In [ ]:
# Change columns, doesn't allow period.

# df_sr_and_nlst = df_sr_and_nlst.rename(columns = {'findingSite.CodeValue': 'FindingSite_CodeValue',
#                                                   'findingSite.CodingSchemeDesignator': 'FindingSite_CodingSchemeDesignator',
#                                                   'findingSite.CodeMeaning': 'FindingSite_CodeMeaning',
#                                                   'finding.CodeValue': 'FindingType_CodeValue',
#                                                   'finding.CodingSchemeDesignator': 'FindingType_CodingSchemeDesignator',
#                                                   'finding.CodeMeaning': 'FindingType_CodeMeaning'})


In [ ]:
df_sr_and_nlst = df_sr_and_nlst.drop('FindingType', axis=1)
df_sr_and_nlst = df_sr_and_nlst.drop('FindingSite', axis=1)

In [ ]:
df_sr_and_nlst.to_csv("/content/sybil_sr_and_nlst.csv")

In [ ]:
# I save this as a table in BQ - sybil_sr_and_nlst

In [ ]:
df_sr_and_nlst.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr',
       'SeriesInstanceUID', 'ReferencedSeriesInstanceUID',
       'sop_count_per_series', 'TrackingIdentifier', 'TrackingUID',
       'ReferencedSOPInstanceUID', 'pixel_spacing_x', 'pixel_spacing_y',
       'width', 'height', 'center_x', 'center_y', 'ipp0', 'ipp1', 'ipp2',
       'sct_slice_num', 'sct_epi_loc', 'sct_margins', 'sct_pre_att', 'de_stag',
       'de_type', 'de_stag_mapping', 'de_type_mapping', 'sct_epi_loc_mapping',
       'sct_margins_mapping', 'sct_pre_att_mapping', 'FindingType_CodeValue',
       'FindingType_CodingSchemeDesignator', 'FindingType_CodeMeaning',
       'FindingSite_CodeValue', 'FindingSite_CodingSchemeDesignator',
       'FindingSite_CodeMeaning'],
      dtype='object')

# B. Join SR bounding boxes with clinical metadata from IDC - keep all bounding boxes even if no clinical metadata

## df_sr - get the metadata from the SRs

In [ ]:
# df_sr = pd.read_csv("/content/bbox_measurements.csv") # later, query instead.

# Query from BQ instead
client_bq = bigquery.Client(project=project_name)
query = f"""
    SELECT
      *
    FROM
      `idc-external-018.sr_nlst_sybil.bbox_measurements`
      """
df_sr = client_bq.query(query).to_dataframe()

In [ ]:
# First add columns for the width, height, center_x, and center_y

width_list = []
height_list = []
center_x_list = []
center_y_list = []

for index, row in df_sr.iterrows():
  # Get values
  x0 = row['x0']; y0 = row['y0']
  x1 = row['x1']; y1 = row['y1']
  x2 = row['x2']; y2 = row['y2']
  x3 = row['x3']; y3 = row['y3']
  # calculate the width, height and center, as these are needed for display
  min_x = np.min([x0, x1, x2, x3]) # using roi.GraphicData: min_x = np.min([bbox[0], bbox[2], bbox[4], bbox[6]])
  max_x = np.max([x0, x2, x2, x3]) # using roi.GraphicData: max_x = np.max([bbox[0], bbox[2], bbox[4], bbox[6]])
  min_y = np.min([y0, y1, y2, y3]) # using roi.GraphicData: min_y = np.min([bbox[1], bbox[3], bbox[5], bbox[7]])
  max_y = np.max([y0, y1, y2, y3]) # using roi.GraphicData: max_y = np.max([bbox[1], bbox[3], bbox[5], bbox[7]])
  width = max_x - min_x
  height = max_y - min_y
  center_x = min_x + width/2
  center_y = min_y + height/2
  # append
  width_list.append(width)
  height_list.append(height)
  center_x_list.append(center_x)
  center_y_list.append(center_y)

# Add columns
df_sr['width'] = width_list
df_sr['height'] = height_list
df_sr['center_x'] = center_x_list
df_sr['center_y'] = center_y_list

df_sr.head()

,PatientID,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,ReferencedSeriesInstanceUID,trackingIdentifier,trackingUniqueIdentifier,finding,findingSite,ReferencedSOPInstanceUID,...,x1,y1,x2,y2,x3,y3,width,height,center_x,center_y
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,1,1.2.826.0.1.3680043.8.498.89537880202650470788...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.15782756894992033517041215...,...,221.213928,285.038818,221.213928,324.776123,176.891541,324.776123,44.322388,39.737305,199.052734,304.907471
1,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,2,1.2.826.0.1.3680043.8.498.38187982882062875892...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.20401816777450596174524730...,...,220.000000,287.000000,220.000000,326.000000,177.000000,326.000000,43.000000,39.000000,198.500000,306.500000
2,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,3,1.2.826.0.1.3680043.8.498.79754527055430472666...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.32049018332506010968782808...,...,220.000000,289.000000,220.000000,327.000000,178.000000,327.000000,42.000000,38.000000,199.000000,308.000000
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,4,1.2.826.0.1.3680043.8.498.60746621313354591972...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.23447016361362557671014824...,...,220.000000,292.000000,220.000000,329.000000,179.000000,329.000000,41.000000,37.000000,199.500000,310.500000
4,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.826.0.1.3680043.8.498.28671036785305828632...,1.2.840.113654.2.55.24023112856488152536348979...,5,1.2.826.0.1.3680043.8.498.58936828276024984484...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.29202164341698057454176258...,...,219.000000,294.000000,219.000000,329.000000,179.000000,329.000000,40.000000,35.000000,199.000000,311.500000


In [ ]:
# We need the following fields in order to convert from pixel coordintes to mm for the bounding box
# And we need the Dimensions, Pixel spacing IPP, especially IPP[2] for the z value

referenced_sop_instance_uid_list = list(df_sr['ReferencedSOPInstanceUID'].values)

client_bq = bigquery.Client(project=project_name)

query = f"""
    SELECT
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      SOPInstanceUID,
      `Rows` as num_rows,
      `Columns` as num_columns,
      PixelSpacing,
      ImagePositionPatient
    FROM
      `bigquery-public-data.idc_current.dicom_all`
    WHERE
      SOPInstanceUID IN UNNEST(@referenced_sop_instance_uid_list)
    ORDER BY
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      ImagePositionPatient[SAFE_OFFSET(2)]
      """

job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("referenced_sop_instance_uid_list", "STRING", referenced_sop_instance_uid_list)])
df_idc = client_bq.query(query, job_config=job_config).to_dataframe()

In [ ]:
# Reformat the PixelSpacing and the ImagePositionPatient columns

df_idc['pixel_spacing_x'] = [np.float32(f[0]) for f in df_idc['PixelSpacing'].values]
df_idc['pixel_spacing_y'] = [np.float32(f[1]) for f in df_idc['PixelSpacing'].values]
df_idc['ipp0'] = [np.float32(f[0]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp1'] = [np.float32(f[1]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp2'] = [np.float32(f[2]) for f in df_idc['ImagePositionPatient'].values]

df_idc = df_idc[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID',
                 'num_rows', 'num_columns',
                 'pixel_spacing_x', 'pixel_spacing_y',
                 'ipp0', 'ipp1', 'ipp2']]

In [ ]:
df_idc.head()

,PatientID,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,num_rows,num_columns,pixel_spacing_x,pixel_spacing_y,ipp0,ipp1,ipp2
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29991037322734048580038819...,512,512,0.585938,0.585938,-149.707031,-319.707031,-76.400002
1,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.27443508115501826384206327...,512,512,0.585938,0.585938,-149.707031,-319.707031,-78.400002
2,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.16899951679153198601142574...,512,512,0.585938,0.585938,-149.707031,-319.707031,-80.400002
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.17481124987277919843449170...,512,512,0.585938,0.585938,-149.707031,-319.707031,-82.400002
4,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29574962348509387538142601...,512,512,0.585938,0.585938,-149.707031,-319.707031,-84.400002


## df_nlst_metadata - get the associated clinical metadata - for classification

In [ ]:
PatientID_list = sorted(list(set(df_idc['PatientID'].values)))
PatientID_list = [str(f) for f in PatientID_list]
print(len(PatientID_list))

SeriesInstanceUID_list = list(set(df_idc['SeriesInstanceUID'].values))
print(len(SeriesInstanceUID_list))

SOPInstanceUID_list = list(set(df_idc['SOPInstanceUID'].values))
print(len(SOPInstanceUID_list))

581
970
9222


In [ ]:
# Here we get the staging data

query = f"""
WITH dicom_mapped AS (
  SELECT
    PatientID,
    StudyInstanceUID,
    StudyDate,
    SeriesInstanceUID,
    SOPInstanceUID,
    InstanceNumber,
    `Rows`,
    `Columns`,
    CASE StudyDate
      WHEN '1999-01-02' THEN 0
      WHEN '2000-01-02' THEN 1
      WHEN '2001-01-02' THEN 2
      ELSE 3
    END AS StudyDate_mapped,
    COUNT(*) OVER (PARTITION BY SeriesInstanceUID) AS sop_count_per_series
  FROM `bigquery-public-data.idc_current.dicom_all`
  WHERE SeriesInstanceUID IN UNNEST(@SeriesInstanceUID_list)
)

SELECT
  dicom_mapped.PatientID AS PatientID,
  dicom_mapped.StudyInstanceUID,
  dicom_mapped.StudyDate,
  dicom_mapped.SeriesInstanceUID,
  dicom_mapped.SOPInstanceUID,
  ctab.sct_slice_num,
  ctab.sct_epi_loc,
  ctab.sct_margins,
  ctab.sct_pre_att,
  ctab.study_yr,
  dicom_mapped.Rows,
  dicom_mapped.Columns,
  dicom_mapped.sop_count_per_series,
  prsn.de_stag,
  CAST(prsn.de_type AS STRING) as de_type
FROM
  `bigquery-public-data.idc_current_clinical.nlst_ctab` AS ctab
JOIN
  `bigquery-public-data.idc_current_clinical.nlst_prsn` AS prsn
  ON prsn.dicom_patient_id = ctab.dicom_patient_id
JOIN
  dicom_mapped
ON dicom_mapped.InstanceNumber = ctab.sct_slice_num
   AND ctab.study_yr = dicom_mapped.StudyDate_mapped
   AND dicom_mapped.PatientID = ctab.dicom_patient_id

"""
job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("SOPInstanceUID_list", "STRING", SOPInstanceUID_list),
                                                       bigquery.ArrayQueryParameter("SeriesInstanceUID_list", "STRING", SeriesInstanceUID_list)])
df_nlst_metadata = client_bq.query(query, job_config=job_config).to_dataframe()

In [ ]:
df_nlst_metadata.head()

,PatientID,StudyInstanceUID,StudyDate,SeriesInstanceUID,SOPInstanceUID,sct_slice_num,sct_epi_loc,sct_margins,sct_pre_att,study_yr,Rows,Columns,sop_count_per_series,de_stag,de_type
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.23447016361362557671014824...,38,1,2,1,0,512,512,162,110,8140
1,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1.2.840.113654.2.55.50761756412482430061802871...,1.2.840.113654.2.55.23050894131215950278743827...,39,1,1,2,1,512,512,157,110,8140
2,100147,1.2.840.113654.2.55.13303292650860633016545772...,1999-01-02,1.2.840.113654.2.55.24785488463405747713776937...,1.2.840.113654.2.55.17981552983033538571417359...,88,1,3,2,0,512,512,110,110,8140
3,100147,1.2.840.113654.2.55.31958452963320032523273261...,2000-01-02,1.2.840.113654.2.55.15708941008648745210499888...,1.2.840.113654.2.55.19851369674160913596156868...,92,1,3,2,1,512,512,116,110,8140
4,100158,1.2.840.113654.2.55.81185422866512279860334872...,2001-01-02,1.2.840.113654.2.55.31060976780967844152296392...,1.2.840.113654.2.55.18324898484332559595555356...,57,1,2,1,2,512,512,146,110,8140


In [ ]:
len(df_nlst_metadata)

1405

In [ ]:
df_nlst_metadata.to_csv("/content/nlst_metadata.csv")

## Join the tables to hold the SR info and clinical metadata info

In [ ]:
print(len(df_sr))

9280


In [ ]:
print(len(df_idc))

9222


In [ ]:
print(len(df_nlst_metadata))

1405


In [ ]:
# Then join with df_idc
df_sr_join = df_sr.merge(df_idc,
                         left_on=['ReferencedSOPInstanceUID'],
                         right_on=['SOPInstanceUID'],
                         suffixes=('','_right'))
# Drop the duplicate column from the right dataframe
df_sr_join = df_sr_join.drop(columns=['PatientID_right','SOPInstanceUID_right'])

print(len(df_sr_join))

9280


In [ ]:
df_sr_join.columns

Index(['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID',
       'ReferencedSeriesInstanceUID', 'trackingIdentifier',
       'trackingUniqueIdentifier', 'finding', 'findingSite',
       'ReferencedSOPInstanceUID', 'ConceptNameCodeSequence', 'GraphicType',
       'x0', 'y0', 'x1', 'y1', 'x2', 'y2', 'x3', 'y3', 'width', 'height',
       'center_x', 'center_y', 'StudyInstanceUID_right',
       'SeriesInstanceUID_right', 'num_rows', 'num_columns', 'pixel_spacing_x',
       'pixel_spacing_y', 'ipp0', 'ipp1', 'ipp2'],
      dtype='object')

In [ ]:
df_nlst_metadata.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'SeriesInstanceUID',
       'SOPInstanceUID', 'sct_slice_num', 'sct_epi_loc', 'sct_margins',
       'sct_pre_att', 'study_yr', 'Rows', 'Columns', 'sop_count_per_series',
       'de_stag', 'de_type'],
      dtype='object')

In [ ]:
# First add the StudyDate, study_yr and sop_count_per_series to the df_sr_join pandas dataframe

df_nlst_metadata_partial = df_nlst_metadata[['PatientID', 'StudyInstanceUID', 'StudyDate', 'SeriesInstanceUID', 'sop_count_per_series']]
df_nlst_metadata_partial = df_nlst_metadata_partial.drop_duplicates()
df_sr_join_partial = df_sr_join.merge(df_nlst_metadata_partial,
                                      left_on=['ReferencedSeriesInstanceUID'],
                                      right_on=['SeriesInstanceUID'],
                                      suffixes=('', '_right'),
                                      how='left')
# Add in study_yr columns to df_sr_join_partial

print(len(df_sr_join_partial))


9280


In [ ]:
# Then join with the df_nlst_metadata
df_sr_and_nlst = df_sr_join_partial.merge(df_nlst_metadata,
                                          left_on=['ReferencedSOPInstanceUID'],
                                          right_on=['SOPInstanceUID'],
                                          suffixes=('','_right'),
                                          how="left")
df_sr_and_nlst = df_sr_and_nlst.drop(columns=['PatientID_right','StudyInstanceUID_right','SeriesInstanceUID_right','num_rows', 'num_columns'])
print(len(df_sr_and_nlst))


df_sr_and_nlst = df_sr_and_nlst.rename({'trackingIdentifier': 'TrackingIdentifier',
                                        'trackingUniqueIdentifier':'TrackingUID',
                                        'finding':'FindingType',
                                        'findingSite': 'FindingSite'}, axis=1)
# Reorder the columns
df_sr_and_nlst = df_sr_and_nlst[['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr', 'SeriesInstanceUID', 'ReferencedSeriesInstanceUID', 'sop_count_per_series',
                                 'TrackingIdentifier', 'TrackingUID', 'ReferencedSOPInstanceUID',
                                 'FindingType', 'FindingSite',
                                 'pixel_spacing_x', 'pixel_spacing_y',
                                 'width', 'height', 'center_x', 'center_y', 'ipp0', 'ipp1', 'ipp2',
                                 'sct_slice_num', 'sct_epi_loc', 'sct_margins', 'sct_pre_att', 'de_stag', 'de_type']]
# Order the values in the columns
df_sr_and_nlst = df_sr_and_nlst.sort_values(by=['PatientID', 'study_yr', 'StudyInstanceUID', 'ReferencedSeriesInstanceUID', 'TrackingIdentifier'])

# Not sure why the study_yr is wrong, can redo.
study_yr_mapping_dict = {datetime.datetime(1999,1,2): np.int32(0),
                         datetime.datetime(2000,1,2): np.int32(1),
                         datetime.datetime(2001,1,2): np.int32(2)}
# study_yr_mapping_dict = {
#     '19990102': 0,
#     '20000102': 1,
#     '20010102': 2
# }
df_sr_and_nlst['StudyDate'] = pd.to_datetime(df_sr_and_nlst['StudyDate'], format='%Y%m%d')
df_sr_and_nlst['study_yr'] = df_sr_and_nlst['StudyDate'].map(study_yr_mapping_dict)
# df_sr_and_nlst['study_yr'] = [np.int32(f) for f in df_sr_and_nlst['study_yr'].values]

print(len(df_sr_and_nlst))

9295
9295


In [ ]:
df_sr_and_nlst.head()

,PatientID,StudyInstanceUID,StudyDate,study_yr,SeriesInstanceUID,ReferencedSeriesInstanceUID,sop_count_per_series,TrackingIdentifier,TrackingUID,ReferencedSOPInstanceUID,...,center_y,ipp0,ipp1,ipp2,sct_slice_num,sct_epi_loc,sct_margins,sct_pre_att,de_stag,de_type
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,4,1.2.826.0.1.3680043.8.498.60746621313354591972...,1.2.840.113654.2.55.23447016361362557671014824...,...,310.500000,-149.707031,-319.707031,-92.400002,38,1,2,1,110,8140
17,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1.0,1.2.826.0.1.3680043.8.498.83730668932055449360...,1.2.840.113654.2.55.50761756412482430061802871...,157,6,1.2.826.0.1.3680043.8.498.45402182495842328878...,1.2.840.113654.2.55.23050894131215950278743827...,...,328.500000,-133.726562,-309.726562,1220.699951,39,1,1,2,110,8140
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,1,1.2.826.0.1.3680043.8.498.89537880202650470788...,1.2.840.113654.2.55.15782756894992033517041215...,...,304.907471,-149.707031,-319.707031,-98.400002,NaN,NaN,NaN,NaN,NaN,NaN
9,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,10,1.2.826.0.1.3680043.8.498.59834539607512637688...,1.2.840.113654.2.55.16899951679153198601142574...,...,320.000000,-149.707031,-319.707031,-80.400002,NaN,NaN,NaN,NaN,NaN,NaN
10,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,11,1.2.826.0.1.3680043.8.498.96263311181661196304...,1.2.840.113654.2.55.27443508115501826384206327...,...,321.500000,-149.707031,-319.707031,-78.400002,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_sr_and_nlst.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr',
       'SeriesInstanceUID', 'ReferencedSeriesInstanceUID',
       'sop_count_per_series', 'TrackingIdentifier', 'TrackingUID',
       'ReferencedSOPInstanceUID', 'FindingType', 'FindingSite',
       'pixel_spacing_x', 'pixel_spacing_y', 'width', 'height', 'center_x',
       'center_y', 'ipp0', 'ipp1', 'ipp2', 'sct_slice_num', 'sct_epi_loc',
       'sct_margins', 'sct_pre_att', 'de_stag', 'de_type'],
      dtype='object')

In [ ]:
# df_sr_and_nlst["study_yr"].replace("<NA>", np.nan, inplace=True)
# df_sr_and_nlst["study_yr"] = (
#     df_sr_and_nlst.groupby("ReferencedSOPInstanceUID")["study_yr"]
#     .transform(lambda g: g.ffill().bfill())
# )

In [ ]:
df_sr_and_nlst.head()

,PatientID,StudyInstanceUID,StudyDate,study_yr,SeriesInstanceUID,ReferencedSeriesInstanceUID,sop_count_per_series,TrackingIdentifier,TrackingUID,ReferencedSOPInstanceUID,...,center_y,ipp0,ipp1,ipp2,sct_slice_num,sct_epi_loc,sct_margins,sct_pre_att,de_stag,de_type
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,4,1.2.826.0.1.3680043.8.498.60746621313354591972...,1.2.840.113654.2.55.23447016361362557671014824...,...,310.500000,-149.707031,-319.707031,-92.400002,38,1,2,1,110,8140
17,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1.0,1.2.826.0.1.3680043.8.498.83730668932055449360...,1.2.840.113654.2.55.50761756412482430061802871...,157,6,1.2.826.0.1.3680043.8.498.45402182495842328878...,1.2.840.113654.2.55.23050894131215950278743827...,...,328.500000,-133.726562,-309.726562,1220.699951,39,1,1,2,110,8140
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,1,1.2.826.0.1.3680043.8.498.89537880202650470788...,1.2.840.113654.2.55.15782756894992033517041215...,...,304.907471,-149.707031,-319.707031,-98.400002,NaN,NaN,NaN,NaN,NaN,NaN
9,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,10,1.2.826.0.1.3680043.8.498.59834539607512637688...,1.2.840.113654.2.55.16899951679153198601142574...,...,320.000000,-149.707031,-319.707031,-80.400002,NaN,NaN,NaN,NaN,NaN,NaN
10,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,11,1.2.826.0.1.3680043.8.498.96263311181661196304...,1.2.840.113654.2.55.27443508115501826384206327...,...,321.500000,-149.707031,-319.707031,-78.400002,NaN,NaN,NaN,NaN,NaN,NaN


## Calculate extra features

In [ ]:
segments_code_mapping_df = pd.read_csv(
    "https://raw.githubusercontent.com/deepakri201/SEG_and_SR_for_NLSTSeg/main/NLSTSeg_icd03_codes.csv"
)

In [ ]:
# segments_code_mapping_df = segments_code_mapping_df[['SegmentedPropertyTypeCodeSequence.CodeValue','SegmentedPropertyTypeCodeSequence.CodeMeaning']]
# segments_code_mapping_df = segments_code_mapping_df.rename(columns={'SegmentedPropertyTypeCodeSequence.CodeValue': 'ICD-03',
#                                                                     'SegmentedPropertyTypeCodeSequence.CodeMeaning': 'Description'})
# segments_code_mapping_df['ICD03'] = [str(f) for f in segments_code_mapping_df['ICD-03'].values]
segments_code_mapping_df = segments_code_mapping_df[['SegmentedPropertyTypeCodeSequence.CodeValue','SegmentedPropertyTypeCodeSequence.CodeMeaning']]
segments_code_mapping_df = segments_code_mapping_df.rename(columns={'SegmentedPropertyTypeCodeSequence.CodeValue': 'ICD03',
                                                                    'SegmentedPropertyTypeCodeSequence.CodeMeaning': 'Description'})
segments_code_mapping_df['ICD03'] = [str(f) for f in segments_code_mapping_df['ICD03'].values]

In [ ]:
segments_code_mapping_df

,ICD03,Description
0,8481,Mucin-producing adenocarcinoma
1,8490,Signet ring cell carcinoma
2,8071,"Squamous cell carcinoma, keratinizing"
3,8252,"Bronchiolo-alveolar carcinoma, non-mucinous"
4,8570,Adenocarcinoma with squamous metaplasia
5,8323,Mixed cell adenocarcinoma
6,8010,Carcinoma in situ
7,8050,Papillary adenocarcinoma
8,8042,Oat cell carcinoma
9,8254,"Bronchiolo-alveolar carcinoma, mixed mucinous ..."


In [ ]:
de_stag_mapping = {'110': "Stage IA",
                   '120': "Stage IB",
                   '210': "Stage IIA",
                   '220': "Stage IIB",
                   '310': "Stage IIIA",
                   '320': "Stage IIIB",
                   '400': "Stage IV",
                   '888': "TNM not available",
                   '900': "Occult Carcinoma",
                   '994': "Carcinoid, cannot be assessed",
                   '999': "Unknown, cannot be assessed"}
# de_type_mapping = segments_code_mapping_df.to_dict(orient='dict')
de_type_mapping = dict(zip(segments_code_mapping_df.ICD03,segments_code_mapping_df.Description))
sct_epi_loc_mapping = {'1': "Right Upper Lobe" ,
                       '2': "Right Middle Lobe",
                       '3': "Right Lower Lobe",
                       '4': "Left Upper Lobe",
                       '5': "Lingula",
                       '6': "Left Lower Lobe",
                       '8': "Other (Specify in comments)"}
sct_margins_mapping = {'1': "Spiculated (Stellate)",
                       '2': "Smooth",
                       '3': "Poorly defined",
                       '9': "Unable to determine"}
sct_pre_att_mapping = {'.M': "Missing",
                       'N': "Not applicable (sct_ab_desc is not 51)",
                       '1': "Soft Tissue",
                       '2': "Ground glass",
                       '3': "Mixed",
                       '4': "Fluid/water",
                       '6': "Fat",
                       '7': "Other",
                       '9': "Unable to determine"}

In [ ]:
df_sr_and_nlst['de_stag_mapping'] = df_sr_and_nlst['de_stag'].map(de_stag_mapping)
df_sr_and_nlst['de_type_mapping'] = df_sr_and_nlst['de_type'].map(de_type_mapping)
df_sr_and_nlst['sct_epi_loc_mapping'] = df_sr_and_nlst['sct_epi_loc'].map(sct_epi_loc_mapping)
df_sr_and_nlst['sct_margins_mapping'] = df_sr_and_nlst['sct_margins'].map(sct_margins_mapping)
df_sr_and_nlst['sct_pre_att_mapping'] = df_sr_and_nlst['sct_pre_att'].map(sct_pre_att_mapping)


In [ ]:
df_sr_and_nlst.head()

,PatientID,StudyInstanceUID,StudyDate,study_yr,SeriesInstanceUID,ReferencedSeriesInstanceUID,sop_count_per_series,TrackingIdentifier,TrackingUID,ReferencedSOPInstanceUID,...,sct_epi_loc,sct_margins,sct_pre_att,de_stag,de_type,de_stag_mapping,de_type_mapping,sct_epi_loc_mapping,sct_margins_mapping,sct_pre_att_mapping
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,4,1.2.826.0.1.3680043.8.498.60746621313354591972...,1.2.840.113654.2.55.23447016361362557671014824...,...,1,2,1,110,8140,Stage IA,Adenocarcinoma,Right Upper Lobe,Smooth,Soft Tissue
17,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1.0,1.2.826.0.1.3680043.8.498.83730668932055449360...,1.2.840.113654.2.55.50761756412482430061802871...,157,6,1.2.826.0.1.3680043.8.498.45402182495842328878...,1.2.840.113654.2.55.23050894131215950278743827...,...,1,1,2,110,8140,Stage IA,Adenocarcinoma,Right Upper Lobe,Spiculated (Stellate),Ground glass
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,1,1.2.826.0.1.3680043.8.498.89537880202650470788...,1.2.840.113654.2.55.15782756894992033517041215...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,10,1.2.826.0.1.3680043.8.498.59834539607512637688...,1.2.840.113654.2.55.16899951679153198601142574...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0.0,1.2.826.0.1.3680043.8.498.39151392084187769915...,1.2.840.113654.2.55.24023112856488152536348979...,162,11,1.2.826.0.1.3680043.8.498.96263311181661196304...,1.2.840.113654.2.55.27443508115501826384206327...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_sr_and_nlst.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr',
       'SeriesInstanceUID', 'ReferencedSeriesInstanceUID',
       'sop_count_per_series', 'TrackingIdentifier', 'TrackingUID',
       'ReferencedSOPInstanceUID', 'FindingType', 'FindingSite',
       'pixel_spacing_x', 'pixel_spacing_y', 'width', 'height', 'center_x',
       'center_y', 'ipp0', 'ipp1', 'ipp2', 'sct_slice_num', 'sct_epi_loc',
       'sct_margins', 'sct_pre_att', 'de_stag', 'de_type', 'de_stag_mapping',
       'de_type_mapping', 'sct_epi_loc_mapping', 'sct_margins_mapping',
       'sct_pre_att_mapping'],
      dtype='object')

In [ ]:
# # Need to split the Finding and FindingSite into columns for the CodeValue, CodingSchemeDesignator, and CodeMeaning, in order to save as csv and be able to load as a BQ table.
# df_expanded = pd.json_normalize(df_sr_and_nlst["FindingType"].apply(json.loads))
# df_sr_and_nlst = df_sr_and_nlst.join(df_expanded)

# df_expanded = pd.json_normalize(df_sr_and_nlst["FindingSite"].apply(json.loads))
# df_sr_and_nlst = df_sr_and_nlst.join(df_expanded)

df_expanded = pd.json_normalize(df_sr_and_nlst["FindingType"])
df_sr_and_nlst = df_sr_and_nlst.join(df_expanded, lsuffix="FindingType")
df_sr_and_nlst = df_sr_and_nlst.rename(columns = {'CodeValue': 'FindingType_CodeValue',
                                                  'CodingSchemeDesignator': 'FindingType_CodingSchemeDesignator',
                                                  'CodeMeaning': 'FindingType_CodeMeaning'})

df_expanded = pd.json_normalize(df_sr_and_nlst["FindingSite"])
df_sr_and_nlst = df_sr_and_nlst.join(df_expanded, lsuffix="FindingSite")
df_sr_and_nlst = df_sr_and_nlst.rename(columns = {'CodeValue': 'FindingSite_CodeValue',
                                                  'CodingSchemeDesignator': 'FindingSite_CodingSchemeDesignator',
                                                  'CodeMeaning': 'FindingSite_CodeMeaning'})

In [ ]:
df_sr_and_nlst.columns

Index(['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr',
       'SeriesInstanceUID', 'ReferencedSeriesInstanceUID',
       'sop_count_per_series', 'TrackingIdentifier', 'TrackingUID',
       'ReferencedSOPInstanceUID', 'FindingType', 'FindingSite',
       'pixel_spacing_x', 'pixel_spacing_y', 'width', 'height', 'center_x',
       'center_y', 'ipp0', 'ipp1', 'ipp2', 'sct_slice_num', 'sct_epi_loc',
       'sct_margins', 'sct_pre_att', 'de_stag', 'de_type', 'de_stag_mapping',
       'de_type_mapping', 'sct_epi_loc_mapping', 'sct_margins_mapping',
       'sct_pre_att_mapping', 'FindingType_CodeValue',
       'FindingType_CodingSchemeDesignator', 'FindingType_CodeMeaning',
       'FindingSite_CodeValue', 'FindingSite_CodingSchemeDesignator',
       'FindingSite_CodeMeaning'],
      dtype='object')

In [ ]:
# Change columns, doesn't allow period.

# df_sr_and_nlst = df_sr_and_nlst.rename(columns = {'findingSite.CodeValue': 'FindingSite_CodeValue',
#                                                   'findingSite.CodingSchemeDesignator': 'FindingSite_CodingSchemeDesignator',
#                                                   'findingSite.CodeMeaning': 'FindingSite_CodeMeaning',
#                                                   'finding.CodeValue': 'FindingType_CodeValue',
#                                                   'finding.CodingSchemeDesignator': 'FindingType_CodingSchemeDesignator',
#                                                   'finding.CodeMeaning': 'FindingType_CodeMeaning'})


In [ ]:
df_sr_and_nlst = df_sr_and_nlst.drop('FindingType', axis=1)
df_sr_and_nlst = df_sr_and_nlst.drop('FindingSite', axis=1)

In [ ]:
df_sr_and_nlst.to_csv("/content/sybil_sr_and_nlst_all_boxes.csv")

In [ ]:
# I save this as a table in BQ - sybil_sr_and_nlst